In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC ## Quality Checks — Verify ETL Output

# COMMAND ----------

from pyspark.sql import functions as F

storage_key = dbutils.secrets.get(scope="retailrocket", key="storage-key")
spark.conf.set("fs.azure.account.key.stdportfolio.blob.core.windows.net", storage_key)

# COMMAND ----------

# MAGIC %md
# MAGIC ### Load Gold Tables

# COMMAND ----------

# Refresh Delta table cache
spark.catalog.refreshTable("gold.dim_date")
spark.catalog.refreshTable("gold.dim_users")
spark.catalog.refreshTable("gold.dim_items")
spark.catalog.refreshTable("gold.dim_categories")
spark.catalog.refreshTable("gold.dim_event_type")
spark.catalog.refreshTable("gold.fact_events")
spark.catalog.refreshTable("silver.events")

dim_date = spark.table("gold.dim_date")
dim_users = spark.table("gold.dim_users")
dim_items = spark.table("gold.dim_items")
dim_categories = spark.table("gold.dim_categories")
dim_event_type = spark.table("gold.dim_event_type")
fact_events = spark.table("gold.fact_events")

# Also load silver for comparison
silver_events = spark.table("silver.events")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 1. Row Count Checks

# COMMAND ----------

print("=" * 50)
print("1. ROW COUNT CHECKS")
print("=" * 50)

checks_passed = 0
checks_failed = 0

# All tables should have rows
tables = {
    "dim_date": dim_date,
    "dim_users": dim_users,
    "dim_items": dim_items,
    "dim_categories": dim_categories,
    "dim_event_type": dim_event_type,
    "fact_events": fact_events,
}

for name, df in tables.items():
    count = df.count()
    status = "PASS" if count > 0 else "FAIL"
    if status == "PASS":
        checks_passed += 1
    else:
        checks_failed += 1
    print(f"  {status} | {name}: {count:,} rows")

# Fact rows should match silver (1:1 grain)
silver_count = silver_events.count()
fact_count = fact_events.count()
status = "PASS" if fact_count == silver_count else "FAIL"
if status == "PASS":
    checks_passed += 1
else:
    checks_failed += 1
print(f"  {status} | fact_events ({fact_count:,}) == silver_events ({silver_count:,})")

# dim_event_type should have exactly 3 types
event_type_count = dim_event_type.count()
status = "PASS" if event_type_count == 3 else "FAIL"
if status == "PASS":
    checks_passed += 1
else:
    checks_failed += 1
print(f"  {status} | dim_event_type: {event_type_count} types (expected 3)")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 2. Null Foreign Key Checks

# COMMAND ----------

print("=" * 50)
print("2. NULL FOREIGN KEY CHECKS")
print("=" * 50)

null_checks = {
    "date_sk": fact_events.filter(F.col("date_sk").isNull()).count(),
    "user_sk": fact_events.filter(F.col("user_sk").isNull()).count(),
    "item_sk": fact_events.filter(F.col("item_sk").isNull()).count(),
    "event_type_sk": fact_events.filter(F.col("event_type_sk").isNull()).count(),
}

for col, null_count in null_checks.items():
    status = "PASS" if null_count == 0 else "FAIL"
    if status == "PASS":
        checks_passed += 1
    else:
        checks_failed += 1
    print(f"  {status} | {col}: {null_count:,} nulls")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 3. Unique Key Checks

# COMMAND ----------

print("=" * 50)
print("3. UNIQUE KEY CHECKS")
print("=" * 50)

unique_checks = {
    "dim_date.date_sk": (dim_date, "date_sk"),
    "dim_users.user_sk": (dim_users, "user_sk"),
    "dim_items.item_sk": (dim_items, "item_sk"),
    "dim_categories.category_sk": (dim_categories, "category_sk"),
    "dim_event_type.event_type_sk": (dim_event_type, "event_type_sk"),
    "fact_events.event_sk": (fact_events, "event_sk"),
}

for label, (df, col) in unique_checks.items():
    total = df.count()
    distinct = df.select(col).distinct().count()
    status = "PASS" if total == distinct else "FAIL"
    if status == "PASS":
        checks_passed += 1
    else:
        checks_failed += 1
    print(f"  {status} | {label}: {total:,} total, {distinct:,} distinct")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 4. Referential Integrity

# COMMAND ----------

print("=" * 50)
print("4. REFERENTIAL INTEGRITY CHECKS")
print("=" * 50)

# Check fact rows have matching dim rows
orphan_checks = {
    "date_sk → dim_date": (fact_events, dim_date, "date_sk"),
    "user_sk → dim_users": (fact_events, dim_users, "user_sk"),
    "item_sk → dim_items": (fact_events, dim_items, "item_sk"),
    "event_type_sk → dim_event_type": (fact_events, dim_event_type, "event_type_sk"),
}


for label, (fact, dim, key) in orphan_checks.items():
    orphans = fact.join(dim, key, "left_anti").count()
    status = "PASS" if orphans == 0 else "FAIL"
    if status == "PASS":
        checks_passed += 1
    else:
        checks_failed += 1
    print(f"  {status} | {label}: {orphans:,} orphans")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 5. Business Logic Checks

# COMMAND ----------

print("=" * 50)
print("5. BUSINESS LOGIC CHECKS")
print("=" * 50)

# 5a. Event types should be view, addtocart, transaction
valid_types = dim_event_type.select("event_type").distinct().collect()
valid_type_names = [r["event_type"] for r in valid_types]
expected_types = {"view", "addtocart", "transaction"}
status = "PASS" if set(valid_type_names) == expected_types else "FAIL"
if status == "PASS":
    checks_passed += 1
else:
    checks_failed += 1
print(f"  {status} | Event types: {valid_type_names}")

# 5b. No future dates in dim_date
future_dates = dim_date.filter(F.col("full_date") > F.current_date()).count()
status = "PASS" if future_dates == 0 else "FAIL"
if status == "PASS":
    checks_passed += 1
else:
    checks_failed += 1
print(f"  {status} | Future dates in dim_date: {future_dates}")

# 5c. Date range should match dataset (2015)
min_date = dim_date.agg(F.min("full_date")).collect()[0][0]
max_date = dim_date.agg(F.max("full_date")).collect()[0][0]
status = "PASS" if min_date.year == 2015 else "FAIL"
if status == "PASS":
    checks_passed += 1
else:
    checks_failed += 1
print(f"  {status} | Date range: {min_date} to {max_date}")

# 5d. Day of week should be 1–7
dow_range = dim_date.agg(
    F.min("day_of_week").alias("min"),
    F.max("day_of_week").alias("max")
).collect()[0]
status = "PASS" if dow_range["min"] >= 1 and dow_range["max"] <= 7 else "FAIL"
if status == "PASS":
    checks_passed += 1
else:
    checks_failed += 1
print(f"  {status} | Day of week range: {dow_range['min']} to {dow_range['max']}")

# 5e. Quarter should be 1–4
q_range = dim_date.agg(
    F.min("quarter").alias("min"),
    F.max("quarter").alias("max")
).collect()[0]
status = "PASS" if q_range["min"] >= 1 and q_range["max"] <= 4 else "FAIL"
if status == "PASS":
    checks_passed += 1
else:
    checks_failed += 1
print(f"  {status} | Quarter range: {q_range['min']} to {q_range['max']}")

# 5f. Users should have at least 1 event
min_events = dim_users.agg(F.min("total_events")).collect()[0][0]
status = "PASS" if min_events >= 1 else "FAIL"
if status == "PASS":
    checks_passed += 1
else:
    checks_failed += 1
print(f"  {status} | Min events per user: {min_events}")

# 5g. No negative item IDs
negative_items = dim_items.filter(F.col("itemid") < 0).count()
status = "PASS" if negative_items == 0 else "FAIL"
if status == "PASS":
    checks_passed += 1
else:
    checks_failed += 1
print(f"  {status} | Negative item IDs: {negative_items}")

# 5h. All 3 event types should exist in fact table
fact_event_types = (
    fact_events
    .join(dim_event_type, "event_type_sk")
    .select("event_type")
    .distinct()
    .count()
)
status = "PASS" if fact_event_types == 3 else "FAIL"
if status == "PASS":
    checks_passed += 1
else:
    checks_failed += 1
print(f"  {status} | Event types in fact: {fact_event_types} (expected 3)")

# 5i. transactionid should only exist for 'transaction' events
# (view and addtocart should have NULL transactionid)
non_txn_with_id = (
    fact_events
    .join(dim_event_type, "event_type_sk")
    .filter(F.col("event_type") != "transaction")
    .filter(F.col("transactionid").isNotNull())
    .count()
)
status = "PASS" if non_txn_with_id == 0 else "FAIL"
if status == "PASS":
    checks_passed += 1
else:
    checks_failed += 1
print(f"  {status} | Non-transaction events with transactionid: {non_txn_with_id}")

# COMMAND ----------

# MAGIC %md
# MAGIC ### 6. Sample Data Inspection

# COMMAND ----------

print("=" * 50)
print("6. SAMPLE DATA INSPECTION")
print("=" * 50)

# 6a. Sample fact events
print("\n--- fact_events (5 rows) ---")
display(fact_events.limit(5))

# 6b. Sample dim_date
print("\n--- dim_date (5 rows) ---")
display(dim_date.limit(5))

# 6c. Sample dim_users (top 5 by total_events)
print("\n--- dim_users (top 5 by events) ---")
display(dim_users.orderBy(F.desc("total_events")).limit(5))

# 6d. Sample dim_items
print("\n--- dim_items (5 rows) ---")
display(dim_items.limit(5))

# 6e. Sample dim_categories
print("\n--- dim_categories (5 rows) ---")
display(dim_categories.limit(5))

# 6f. Sample dim_event_type
print("\n--- dim_event_type (all rows) ---")
display(dim_event_type)

# 6g. Event distribution by type
print("\n--- Event distribution by type ---")
display(
    fact_events
    .join(dim_event_type, "event_type_sk")
    .groupBy("event_type")
    .count()
    .orderBy(F.desc("count"))
)

# 6h. Date coverage (events per month)
print("\n--- Events per month ---")
display(
    fact_events
    .join(dim_date, "date_sk")
    .groupBy("year", "month", "month_name")
    .count()
    .orderBy("year", "month")
)

# COMMAND ----------

# MAGIC %md
# MAGIC ### Summary

# COMMAND ----------

total = checks_passed + checks_failed
print("=" * 50)
print("QUALITY CHECK SUMMARY")
print("=" * 50)
print(f"  Passed: {checks_passed}/{total}")
print(f"  Failed: {checks_failed}/{total}")

if checks_failed == 0:
    print("\n  ALL CHECKS PASSED")
else:
    print(f"\n  {checks_failed} CHECKS FAILED — review above")

In [0]:
# Check if fact_events keys actually match dim keys
fact = spark.table("gold.fact_events")
dim = spark.table("gold.dim_date")

# Show sample mismatches
fact.select("date_sk").distinct().subtract(dim.select("date_sk")).show(5)